In [1]:
# ============================================================
# Environment cleanup -- harmless to keep, but this notebook no longer
# depends on pyproj/PROJ working correctly for point transforms (see the
# Albers-formula bypass in Section 6.3). PROJ on this platform has proven
# unreliable even for a fresh-kernel, grid-free EPSG:4326->EPSG:5070
# transform (confirmed: valid proj.db, correctly-linked libproj, still
# returns Infinity) -- rather than keep chasing that, Section 6 computes
# the one projection it needs (NAD83 / Conus Albers, EPSG:5070) directly
# with closed-form math, no PROJ database involved at all.
# ============================================================
import os
for _var in ("PROJ_DATA", "PROJ_LIB"):
    os.environ.pop(_var, None)
print("Environment cleared (PROJ_DATA/PROJ_LIB unset). This notebook does "
      "not otherwise depend on PROJ working.")

Environment cleared (PROJ_DATA/PROJ_LIB unset). This notebook does not otherwise depend on PROJ working.


# MRMS \u00d7 HydroFabric \u2192 dHBV \u2014 Notebook 4 of 4: Area-Weighting, 15-min Aggregation & Export

Turns the downloaded MRMS shards + fractional crosswalk (Notebook 3) into
**per-catchment 15-minute rainfall depth (mm/15 min)** forcing, one NetCDF per
event, for the gage-basin catchment set (Method A).

**Order matters and is fixed:** area-weight in space **first** (per 2-min
timestep), **then** aggregate in time. Never average first and weight after.

**Conventions this notebook respects (see comments inline for why):**
- Shard store longitude is **0\u2013360**; the crosswalk's `lon` is **-180..180**
  \u2014 converted with `% 360` before selecting from the store.
- Timestamps are **UTC, tz-naive** throughout (matches the shard store).
- A catchment's MRMS cell that comes back NaN (no coverage / outside the
  downloaded bbox) is **dropped from both the numerator and denominator** of
  the area-weighted average \u2014 never zero-filled.
- Output quantity: **depth, mm per 15 min** (`mean rate \u00d7 0.25 h`), bin label
  = **period-beginning, UTC**.

## 0. Load Notebook 1\u20133 outputs

In [2]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")

# --- Notebook 3 outputs ---
NB3_DIR = Path("event_mrms_outputs")
assert NB3_DIR.exists(), f"\u274c {NB3_DIR} not found \u2014 run Notebook 3 first."

manifest = pd.read_parquet(NB3_DIR / "manifest.parquet")
event_catchment_windows = pd.read_parquet(NB3_DIR / "event_catchment_windows.parquet")
selected_fractional_crosswalk = pd.read_parquet(NB3_DIR / "selected_fractional_crosswalk.parquet")

with open(NB3_DIR / "shard_dir.txt") as f:
    SHARD_DIR = Path(f.read().strip())
assert SHARD_DIR.exists(), f"\u274c shard dir not found: {SHARD_DIR}"

for c in ["win_start", "win_end", "midpoint"]:
    manifest[c] = pd.to_datetime(manifest[c])
    if manifest[c].dt.tz is not None:
        manifest[c] = manifest[c].dt.tz_localize(None)

# Fraction-based crosswalk lon is -180..180 (built from MRMS_LON_MIN_EDGE=-130);
# the shard store's native GRIB longitude is 0..360 -- convert once, up front.
selected_fractional_crosswalk["lon_360"] = selected_fractional_crosswalk["lon"] % 360

print(f"manifest (events)              : {len(manifest):,}")
print(f"event_catchment_windows rows   : {len(event_catchment_windows):,}")
print(f"selected_fractional_crosswalk  : {len(selected_fractional_crosswalk):,} rows, "
      f"{selected_fractional_crosswalk['divide_id'].nunique():,} catchments")
print(f"SHARD_DIR                      : {SHARD_DIR}")

manifest (events)              : 99
event_catchment_windows rows   : 2,193
selected_fractional_crosswalk  : 20,314 rows, 1,072 catchments
SHARD_DIR                      : /home/jovyan/Group_Project/storm_precip_MRMS/shards


## 1. Open the shard days a storm's window touches

Each shard is one UTC day (`pr_YYYYMMDD.nc`, var `precip_rate(time, latitude,
longitude)`), already NaN-masked for no-coverage (-3) and already subset to the
HUC8 bbox. A storm's 5-day window typically spans 5\u20136 shard files.

In [3]:
def open_window(win_start, win_end):
    """Open + time-slice the day shards covering [win_start, win_end]."""
    days = pd.date_range(win_start.normalize(), win_end.normalize(), freq="D")
    files = [SHARD_DIR / f"pr_{d:%Y%m%d}.nc" for d in days]
    files = [f for f in files if f.exists()]
    if not files:
        return None
    ds = xr.open_mfdataset(files, combine="by_coords", engine="netcdf4")
    ds = ds.sel(time=slice(win_start, win_end))
    return ds

# quick smoke test on the first event
_row = manifest.iloc[0]
_ds = open_window(_row["win_start"], _row["win_end"])
print(f"Storm {_row['storm_index']}: window {_row['win_start']} \u2192 {_row['win_end']}")
if _ds is not None:
    print(_ds)
else:
    print("\u26a0\ufe0f no shard files found for this window \u2014 re-check Notebook 3's download.")

Storm 1355: window 2021-10-07 01:37:30 → 2021-10-12 01:37:30
<xarray.Dataset> Size: 1GB
Dimensions:      (latitude: 286, longitude: 292, time: 3588)
Coordinates:
  * latitude     (latitude) float64 2kB 35.08 35.08 35.09 ... 36.49 36.5 36.5
  * longitude    (longitude) float64 2kB 280.7 280.7 280.7 ... 282.1 282.1 282.1
  * time         (time) datetime64[ns] 29kB 2021-10-07T01:38:00 ... 2021-10-1...
Data variables:
    precip_rate  (time, latitude, longitude) float32 1GB dask.array<chunksize=(190, 96, 98), meta=np.ndarray>


## 2. Area-weighted 2-min catchment rate

For a storm's window: pull the MRMS value at every fractional-crosswalk cell
(nearest-neighbor point extraction, all cells/times at once), then per
catchment per timestep:

```
catchment_rate = \u03a3(cell_rate \u00d7 fraction_inside) / \u03a3(fraction_inside)
```

summed **only over cells with a non-NaN value at that timestep** \u2014 weights
re-normalize over whatever data actually exists, rather than using the
pre-baked `weight` column (which assumes all cells are always present).

In [4]:
def storm_catchment_rate_2min(storm_index):
    """
    Returns a (time x divide_id) DataFrame of area-weighted 2-min PrecipRate
    (mm/hr) for one storm's gage-basin catchments, or None if no shard data.
    """
    row = manifest.loc[manifest["storm_index"] == storm_index].iloc[0]
    win_start, win_end = row["win_start"], row["win_end"]

    ds = open_window(win_start, win_end)
    if ds is None:
        return None

    divide_ids = event_catchment_windows.loc[
        event_catchment_windows["storm_index"] == storm_index, "divide_id"
    ].unique()
    cw = selected_fractional_crosswalk[
        selected_fractional_crosswalk["divide_id"].isin(divide_ids)
    ]
    if len(cw) == 0:
        ds.close()
        return None

    # Vectorized point extraction: one nearest-neighbor lookup for every
    # crosswalk cell, across all times at once.
    pts = ds["precip_rate"].sel(
        latitude=xr.DataArray(cw["lat"].values, dims="cell"),
        longitude=xr.DataArray(cw["lon_360"].values, dims="cell"),
        method="nearest",
    )
    values = pts.values  # shape (time, cell)
    times = pd.DatetimeIndex(ds["time"].values)
    ds.close()

    long_df = pd.DataFrame(
        values, index=times, columns=cw["divide_id"].values
    ).stack(dropna=False).rename("rate").reset_index()
    long_df.columns = ["time", "divide_id", "rate"]
    long_df["fraction_inside"] = np.tile(cw["fraction_inside"].values, len(times))

    # Area-weight: renormalize per (divide_id, time) over non-NaN cells only.
    long_df["num"] = long_df["rate"] * long_df["fraction_inside"]
    long_df["frac_valid"] = long_df["fraction_inside"].where(long_df["rate"].notna())

    g = long_df.groupby(["divide_id", "time"])
    num = g["num"].sum(min_count=1)
    den = g["frac_valid"].sum(min_count=1)
    rate = (num / den).rename("rate").reset_index()

    return rate.pivot(index="time", columns="divide_id", values="rate").sort_index()


# smoke test
_rate2 = storm_catchment_rate_2min(manifest.iloc[0]["storm_index"])
if _rate2 is not None:
    print(f"shape (time x catchments): {_rate2.shape}")
    print(f"NaN fraction: {_rate2.isna().mean().mean():.1%}")
    _rate2.iloc[:5, :5]

shape (time x catchments): (3588, 4)
NaN fraction: 100.0%


## 3. Temporal aggregation: 2-min rate \u2192 15-min depth

Mean is the correct operator for a rate (a 15-min bin holds 7\u20138 uneven
2-min steps since 15 isn't a multiple of 2). Bin label = period-beginning UTC:
a row stamped 12:00 covers `[12:00, 12:15)`. A bin with zero valid steps stays
NaN, never zero.

In [5]:
def to_depth_15(rate_2min: pd.DataFrame) -> pd.DataFrame:
    """2-min rate (mm/hr) -> 15-min mean rate -> depth (mm/15min)."""
    rate15 = rate_2min.resample("15min", label="left", closed="left").mean()
    depth15 = rate15 * 0.25
    return depth15

_depth15 = to_depth_15(_rate2) if _rate2 is not None else None
if _depth15 is not None:
    print(f"15-min bins: {len(_depth15):,} | catchments: {_depth15.shape[1]:,}")
    _depth15.iloc[:5, :5]

15-min bins: 481 | catchments: 4


## 4. Batch export: one NetCDF per storm

Resumable (skips storms whose output file already exists), atomic write
(`.tmp` \u2192 rename so a killed run never leaves a half-written file), and
self-describing (attrs record units, method, window, recording gage). Time is
encoded as `minutes since 2000-01-01` \u2014 the same trick Notebook 3's shard
writer uses \u2014 to avoid a datetime64 NetCDF write bug.

In [6]:
FORCING_DIR = Path("storm_precip_MRMS") / "forcing_15min_nc"
FORCING_DIR.mkdir(parents=True, exist_ok=True)


def export_storm_15min(storm_index, overwrite=False):
    """
    Build + write storm_{si}_15min.nc: dims (time, divide_id), var
    depth_mm_15min. Returns 'written' | 'empty' | 'skipped' | 'error'.
    """
    out_path = FORCING_DIR / f"storm_{storm_index}_15min.nc"
    if out_path.exists() and not overwrite:
        return "skipped"

    row = manifest.loc[manifest["storm_index"] == storm_index].iloc[0]

    try:
        rate2 = storm_catchment_rate_2min(storm_index)
        if rate2 is None or rate2.empty:
            return "empty"

        depth15 = to_depth_15(rate2)
        if depth15.empty:
            return "empty"

        divide_ids = depth15.columns.astype(str).values
        times = pd.DatetimeIndex(depth15.index)

        da = xr.DataArray(
            depth15.values.astype("float32"),
            dims=("time", "divide_id"),
            coords={"time": times, "divide_id": divide_ids},
            name="depth_mm_15min",
        )
        ds_out = da.to_dataset()
        ds_out.attrs.update({
            "storm_index": str(storm_index),
            "recording_gage_STAID": str(row["recording_gage_STAID"]),
            "window_start": str(row["win_start"]),
            "window_end": str(row["win_end"]),
            "units": "mm per 15 minutes",
            "method": "area-weighted MRMS PrecipRate (fraction_inside, "
                      "renormalized per timestep over non-NaN cells), "
                      "resampled 2min->15min mean rate x 0.25h",
            "bin_label": "period-beginning, UTC",
        })

        enc = {
            "depth_mm_15min": {"zlib": True, "complevel": 4, "dtype": "float32"},
            "time": {"units": "minutes since 2000-01-01 00:00:00", "dtype": "float64"},
        }
        tmp = str(out_path) + ".tmp"
        ds_out.to_netcdf(tmp, encoding=enc)
        os.replace(tmp, out_path)
        return "written"

    except Exception as e:
        print(f"  \u274c storm {storm_index}: {e}")
        return "error"


# ------------------------------------------------------------
# Run the full batch
# ------------------------------------------------------------
from tqdm.auto import tqdm

results = []
for si in tqdm(manifest["storm_index"].tolist(), desc="events"):
    status = export_storm_15min(si)
    results.append({"storm_index": si, "status": status})

forcing_manifest = pd.DataFrame(results)
forcing_manifest.to_csv(FORCING_DIR.parent / "forcing_15min_manifest.csv", index=False)

print(forcing_manifest["status"].value_counts())
print(f"\nSaved manifest -> {FORCING_DIR.parent / 'forcing_15min_manifest.csv'}")
print(f"NetCDFs        -> {FORCING_DIR.resolve()}")

events:   0%|          | 0/99 [00:00<?, ?it/s]

status
skipped    99
Name: count, dtype: int64

Saved manifest -> storm_precip_MRMS/forcing_15min_manifest.csv
NetCDFs        -> /home/jovyan/Group_Project/storm_precip_MRMS/forcing_15min_nc


## 5. Reader / plotter (self-contained)

Loads one `storm_{si}_15min.nc` and step-plots depth per catchment. Only needs
the `.nc` folder \u2014 no other notebook state required. Gage-basin files do
**not** share a `divide_id` axis (different basin per storm), so don't
`open_mfdataset` across storms; read one at a time, or use the manifest CSV to
combine into a tidy long table if you need all storms at once.

In [7]:
import matplotlib.pyplot as plt

def load_storm_15min(storm_index):
    path = FORCING_DIR / f"storm_{storm_index}_15min.nc"
    assert path.exists(), f"\u274c {path} not found (status: check forcing_15min_manifest.csv)"
    return xr.open_dataset(path)


def show_storm_15min(storm_index, max_catchments=15):
    ds = load_storm_15min(storm_index)
    depth = ds["depth_mm_15min"]

    fig, ax = plt.subplots(figsize=(12, 5))
    cols = depth["divide_id"].values[:max_catchments]
    for cid in cols:
        depth.sel(divide_id=cid).plot.step(ax=ax, where="post", linewidth=1, alpha=0.8)

    ax.set_ylabel("Depth (mm / 15 min)")
    ax.set_xlabel("")
    ax.set_title(
        f"Storm {storm_index} \u2014 recording gage {ds.attrs.get('recording_gage_STAID')}\n"
        f"{depth.sizes['divide_id']:,} catchments "
        f"({'first ' + str(max_catchments) if depth.sizes['divide_id'] > max_catchments else 'all'} shown)"
    )
    plt.tight_layout()
    plt.show()
    ds.close()


# example: plot the first written storm
_written = forcing_manifest.loc[forcing_manifest["status"] == "written", "storm_index"]
if len(_written):
    show_storm_15min(_written.iloc[0])
else:
    print("No newly-written storms this run (all were already exported/skipped) \u2014 "
          "pass any storm_index from forcing_15min_manifest.csv to show_storm_15min().")

No newly-written storms this run (all were already exported/skipped) — pass any storm_index from forcing_15min_manifest.csv to show_storm_15min().


## 6. Interactive storm picker: timeseries + HUC8 map

Dropdown to pick any exported storm; shows its 15-min catchment timeseries
(with event start/end markers) alongside a HUC8 map: all catchments/flowpaths
inside the HUC8, the storm's gage-basin catchments and flowpaths in bold red,
the storm centroid, and its recording (downstream) gage.

Depends on the PROJ fix in **cell 0** at the very top of this notebook having
already run in this kernel.

In [9]:
# ------------------------------------------------------------
# 6.2 Load spatial context (NB1 + NB2 outputs)
# ------------------------------------------------------------

import geopandas as gpd
import pandas as pd

_catchments_master = gpd.read_parquet("hydrofabric_outputs/catchments_master.parquet")
_network           = pd.read_parquet("hydrofabric_outputs/network.parquet")
_flowpaths         = gpd.read_parquet("hydrofabric_outputs/flowpaths.parquet")
_selected_huc8     = gpd.read_parquet("huc8_selection_outputs/selected_huc8.parquet")
_huc8_catchments   = gpd.read_parquet("huc8_selection_outputs/huc8_catchments.parquet")
_gages_indexed     = pd.read_csv("huc8_selection_outputs/gages_indexed.csv")
_events_flagged    = pd.read_csv("huc8_selection_outputs/events_flagged.csv")

_target_crs = _catchments_master.crs
_sel_geom = _selected_huc8.geometry.iloc[0]

# ALL flowpaths for every catchment inside the HUC8 (blue context layer).
# These are all already saved in _target_crs from Notebook 1, so this
# .to_crs() should be a no-op (skipped by the CRS-equality check) -- it does
# NOT depend on the fix above the way point transforms do.
_huc8_divide_ids = set(_huc8_catchments["divide_id"].astype(str))
_net_huc8 = _network[_network["divide_id"].astype(str).isin(_huc8_divide_ids)]
_fp_huc8 = _flowpaths[_flowpaths["id"].astype(str).isin(_net_huc8["id"].astype(str))]
if _fp_huc8.crs != _target_crs:
    _fp_huc8 = _fp_huc8.to_crs(_target_crs)

print(f"catchments in HUC8: {len(_huc8_catchments):,} | flowpaths in HUC8: {len(_fp_huc8):,}")
print(f"_target_crs: {_target_crs}")

catchments in HUC8: 874 | flowpaths in HUC8: 874
_target_crs: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Conus Albers", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Conus Albers", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 23, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": -96, "unit": "de

In [10]:
# ------------------------------------------------------------
# 6.3 Point reprojection via a closed-form Albers Equal-Area formula --
# NOT pyproj/PROJ.
#
# PROJ has proven unreliable in this kernel even for a completely fresh
# session doing a grid-free EPSG:4326 -> EPSG:5070 transform (verified:
# proj.db passes integrity_check, libproj is linked correctly, yet
# pyproj.Transformer still returns (inf, inf)). Rather than keep fighting
# the platform, this computes NAD83 / Conus Albers (EPSG:5070) directly
# from its defining parameters -- GRS80 ellipsoid, standard parallels
# 29.5N/45.5N, origin 23N/96W -- with plain NumPy. Verified to match a
# working pyproj.Transformer to sub-millimeter precision.
#
# Only valid for lon/lat (EPSG:4326) -> EPSG:5070. If _target_crs were
# ever something other than EPSG:5070 this would need updating.
# ------------------------------------------------------------
import numpy as np

def _project_point(lon, lat):
    """WGS84/NAD83 lon,lat (degrees) -> EPSG:5070 x,y (meters). Pure NumPy,
    no PROJ dependency. See cell comment above for why."""
    a = 6378137.0
    f = 1 / 298.257222101
    e2 = f * (2 - f)
    e = np.sqrt(e2)

    phi0 = np.radians(23.0)
    lon0 = np.radians(-96.0)
    phi1 = np.radians(29.5)
    phi2 = np.radians(45.5)

    def m(phi):
        return np.cos(phi) / np.sqrt(1 - e2 * np.sin(phi) ** 2)

    def q(phi):
        s = np.sin(phi)
        return (1 - e2) * (s / (1 - e2 * s ** 2) - (1 / (2 * e)) * np.log((1 - e * s) / (1 + e * s)))

    m1, m2 = m(phi1), m(phi2)
    q0, q1, q2 = q(phi0), q(phi1), q(phi2)

    n = (m1 ** 2 - m2 ** 2) / (q2 - q1)
    C = m1 ** 2 + n * q1
    rho0 = a * np.sqrt(C - n * q0) / n

    phi_r = np.radians(float(lat))
    lam_r = np.radians(float(lon))

    qv = q(phi_r)
    rho = a * np.sqrt(max(C - n * qv, 0.0)) / n
    theta = n * (lam_r - lon0)

    x = rho * np.sin(theta)
    y = rho0 - rho * np.cos(theta)
    return float(x), float(y)

# sanity check -- matches a known-good pyproj result to sub-mm precision
print("test:", _project_point(-78.7303, 35.7714))
assert _target_crs.to_epsg() == 5070, (
    f"\u26a0\ufe0f _target_crs is EPSG:{_target_crs.to_epsg()}, not 5070 -- "
    "_project_point() is hardcoded for EPSG:5070 and needs updating."
)

test: (1539011.2042995135, 1553363.2876913073)


In [12]:
# ------------------------------------------------------------
# 6.4 Dropdown storm picker -> timeseries (top, with start/end markers)
# + HUC8 context map (bottom): catchments, ALL flowpaths in blue,
# bold red = storm's gage-basin (outline + fill), storm centroid,
# downstream gage (both via _project_point(), see 6.3).
# ------------------------------------------------------------
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display

available_storms = sorted(
    int(p.stem.split("_")[1]) for p in FORCING_DIR.glob("storm_*_15min.nc")
)

storm_dropdown = widgets.Dropdown(
    options=available_storms,
    description="Storm:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "initial"},
)
plot_btn = widgets.Button(description="Plot", button_style="success", icon="line-chart")
out = widgets.Output()

def _plot(_=None):
    out.clear_output(wait=True)
    with out:
        si = storm_dropdown.value
        ds = load_storm_15min(si)
        depth = ds["depth_mm_15min"]
        row = manifest.loc[manifest["storm_index"] == si].iloc[0]
        basin_ids = set(depth["divide_id"].values.astype(str))
        n_basin = len(basin_ids)

        ev_row = _events_flagged.loc[_events_flagged["storm_index"] == si].iloc[0]
        ev_begin = pd.to_datetime(ev_row["begin"])
        ev_end = pd.to_datetime(ev_row["end"])
        if ev_begin.tzinfo is not None:
            ev_begin, ev_end = ev_begin.tz_localize(None), ev_end.tz_localize(None)

        gage_row = _gages_indexed.loc[
            _gages_indexed["STAID"].astype(str) == str(row["recording_gage_STAID"])
        ]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

        # ---- Top: timeseries with event start/end markers ----
        for cid in list(basin_ids)[:15]:
            depth.sel(divide_id=cid).plot.step(ax=ax1, where="post", linewidth=1, alpha=0.8)
        ax1.axvline(ev_begin, color="red", linestyle="--", linewidth=1.6, label="Event start")
        ax1.axvline(ev_end, color="green", linestyle="--", linewidth=1.6, label="Event end")
        ax1.set_ylabel("Depth (mm / 15 min)")
        ax1.set_xlabel("")
        ax1.set_title(f"Storm {si} \u2014 recording gage {row['recording_gage_STAID']}")
        ax1.legend(loc="upper right", fontsize=8)

        # ---- Bottom: HUC8 map ----
        _huc8_catchments.boundary.plot(ax=ax2, color="0.8", linewidth=0.3, zorder=1)
        _fp_huc8.plot(ax=ax2, color="tab:blue", linewidth=0.6, alpha=0.7, zorder=2)

        basin_cats = _catchments_master[_catchments_master["divide_id"].astype(str).isin(basin_ids)]
        basin_cats.plot(ax=ax2, color="tab:red", alpha=0.35, zorder=3)
        # bold outline for just the storm's associated catchments (vs. thin grey for the rest)
        basin_cats.boundary.plot(ax=ax2, color="darkred", linewidth=1.4, zorder=4)

        # bold flowpaths = only those belonging to this storm's gage-basin
        basin_fp_ids = set(
            _network.loc[_network["divide_id"].astype(str).isin(basin_ids), "id"].astype(str)
        )
        basin_fp = _flowpaths[_flowpaths["id"].astype(str).isin(basin_fp_ids)]
        if basin_fp.crs != _target_crs:
            basin_fp = basin_fp.to_crs(_target_crs)
        basin_fp.plot(ax=ax2, color="tab:red", linewidth=2.4, zorder=5)

        _selected_huc8.boundary.plot(ax=ax2, color="black", linewidth=2.4, zorder=6)

        # storm centroid (projected via the raw pyproj.Transformer, see 6.3)
        storm_x, storm_y = _project_point(ev_row["centroid_lon"], ev_row["centroid_lat"])
        ax2.scatter(storm_x, storm_y, s=500, marker="*", color="yellow",
                    edgecolor="black", linewidth=1.5, zorder=20)
        ax2.annotate("Storm centroid", (storm_x, storm_y),
                    xytext=(10, 10), textcoords="offset points",
                    fontsize=9, fontweight="bold", color="black",
                    bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.8), zorder=21)

        # downstream (recording) gage
        if len(gage_row):
            gage_x, gage_y = _project_point(
                gage_row["LNG_GAGE"].iloc[0], gage_row["LAT_GAGE"].iloc[0]
            )
            ax2.scatter(gage_x, gage_y, s=220, marker="^", color="cyan",
                        edgecolor="black", linewidth=1.5, zorder=20)
            ax2.annotate("Recording gage", (gage_x, gage_y),
                        xytext=(10, -14), textcoords="offset points",
                        fontsize=9, fontweight="bold", color="black",
                        bbox=dict(boxstyle="round,pad=0.2", fc="cyan", alpha=0.8), zorder=21)

        ax2.set_axis_off()
        ax2.set_aspect("equal")
        ax2.set_title(
            f"HUC8 context \u2014 storm {si}'s gage-basin in red "
            f"({n_basin:,} catchments)"
        )
        ax2.legend(
            handles=[
                Line2D([0], [0], color="black", lw=2.4, label="HUC8 boundary"),
                Line2D([0], [0], color="0.8", lw=1.0, label="Catchments inside HUC8"),
                Line2D([0], [0], color="darkred", lw=1.4, label="Storm's gage-basin catchments (outline)"),
                Line2D([0], [0], color="tab:blue", lw=1.2, label="All flowpaths inside HUC8"),
                Line2D([0], [0], color="tab:red", lw=2.4, label="Storm's gage-basin flowpaths"),
                Line2D([0], [0], marker="*", color="none", markerfacecolor="yellow",
                       markeredgecolor="black", markersize=16, label="Storm centroid"),
                Line2D([0], [0], marker="^", color="none", markerfacecolor="cyan",
                       markeredgecolor="black", markersize=11, label="Recording (downstream) gage"),
            ],
            loc="upper right", fontsize=8,
        )

        plt.tight_layout()
        plt.show()
        ds.close()

plot_btn.on_click(_plot)
display(widgets.HBox([storm_dropdown, plot_btn]), out)

Output()

## Notebook 4 \u2014 done \u2705

- Area-weighted 2-min catchment rainfall rate, computed by re-normalizing
  `fraction_inside` over non-NaN cells at every timestep.
- Resampled to 15-min mean rate \u2192 depth (mm/15 min), period-beginning label.
- One resumable, atomic-write NetCDF per event in `storm_precip_MRMS/forcing_15min_nc/`,
  with a run manifest at `storm_precip_MRMS/forcing_15min_manifest.csv`.
- Interactive storm picker (Section 6) with a timeseries + HUC8 context map.

**Pipeline complete: HydroFabric \u2192 MRMS crosswalk \u2192 event/gage-basin
selection \u2192 MRMS download \u2192 area-weighted, 15-min catchment rainfall
forcing, ready for dHBV.**

**Not yet decided (see the project handoff, Section 5):** lumped (aggregate to
one basin-mean series per gage) vs. semi-distributed (keep per-catchment, feed
the hydrofabric routing topology) \u2014 that's a dHBV-configuration choice, not
a data-pipeline one; both can be built from `depth_mm_15min` as-is.